# Training on Kaggle

My laptop only does about 25 steps/s on CPU, so I run the training here on a Kaggle GPU.

Before running: Settings -> Accelerator: GPU T4, and Internet on. To let it run in the background I use Save Version -> Save & Run All, then download the zips from the Output tab.

In [ ]:
!nvidia-smi

I clone into /tmp so the repo and the .venv don't end up in the notebook output, only the result zips do.

In [ ]:
%cd /tmp
!git clone https://github.com/Samkwizera/deep_q_lerning-group-12.git
%cd /tmp/deep_q_lerning-group-12

In [ ]:
import os

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
# kaggle sets this to its inline backend, which the uv venv doesn't have, so matplotlib crashes on import
os.environ["MPLBACKEND"] = "Agg"
!uv sync
# shell commands don't stop the notebook when they fail, so check by hand
if _exit_code != 0:
    raise RuntimeError("uv sync failed, check the output above")
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## Runs

First run is the baseline with the default hyperparameters for 500k steps. I mainly want to see how fast it trains on the GPU and around which step Pong starts improving, so Orrie and I can agree on the step budget for our experiments.

Later I'll just add my experiments to this dict, e.g. `"Ihimbazwe_Samuel/exp02": "--eps-end 0.01 ..."`.

In [ ]:
runs = {
    "baseline": "--total-steps 500000",
}

# -u so the episode logs show up live instead of all at the end
for out_dir, run_args in runs.items():
    !uv run python -u train.py {run_args} --out-dir {out_dir}
    if _exit_code != 0:
        raise RuntimeError(f"{out_dir} failed, check the error above")

In [ ]:
from IPython.display import Image, display

for out_dir in runs:
    print(out_dir)
    display(Image(f"{out_dir}/plot.png"))

In [ ]:
import shutil

for out_dir in runs:
    shutil.make_archive(f"/kaggle/working/{out_dir.replace('/', '_')}", "zip", out_dir)

!ls -lh /kaggle/working